# Applied Unsupervised Learning Techniques

### Imports

In [ ]:
import numpy as np
import altair as alt
import pandas as pd
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering

from birds.source_data import nabbp, avonet

### Configuration

In [ ]:
RANDOM_STATE = 42

In [ ]:
alt.data_transformers.enable("vegafusion")

### Load NABBP Species Data

In [ ]:
def load_nabbp_species() -> pd.DataFrame:
    lookup = nabbp.LookupTables()
    df = lookup.species
    return df[df['ENDANGERED'] == 'Y']
    

nabbp_species_df = load_nabbp_species()
nabbp_species_df.head()

### Load Avonet Bird Tree Data

In [ ]:
def load_avonet_data() -> pd.DataFrame:
    df = avonet.DataTables().bird_tree
    return df

avonet_df = load_avonet_data()
avonet_df.head()

### Merge Avonet and NABBP Datasets

#### Compute dataset overlap using the scientific name column

In [ ]:
def compute_overlap(avonet_data: pd.DataFrame, nabbp_data: pd.DataFrame):
    bt= set(avonet_data['Species3'].unique())
    d = set(nabbp_data['SCI_NAME'].unique())

    overlap  = bt.intersection(d)
    return len(overlap)
    

compute_overlap(avonet_df, nabbp_species_df)

### Transform and Clean Merged Dataset

In [ ]:
def merge_datasets(avonet_data: pd.DataFrame, nabbp_data: pd.DataFrame) -> pd.DataFrame:
    merged_df = pd.merge(avonet_data, nabbp_data, left_on='Species3', right_on='SCI_NAME', how='left')
    return merged_df


merged_df = merge_datasets(avonet_df, nabbp_species_df)
merged_df.head()

In [ ]:
def transform_merge_dataset(merged_data: pd.DataFrame) -> pd.DataFrame:
    avonet_feature_columns = [
        'Beak.Length_Culmen',
        'Beak.Length_Nares',
        'Beak.Width',
        'Beak.Depth',
        'Tarsus.Length',
        'Wing.Length',
        'Kipps.Distance',
        'Hand-Wing.Index',
        'Tail.Length',
        'Mass',
    ]
    id_columns = [
        'SPECIES_ID',
        'Species3',
        'Family3',
        'Order3',
        'ENDANGERED',
    ]
    columns_to_keep = id_columns + avonet_feature_columns
    cleaned = (
        merged_data[columns_to_keep]
            .rename(columns={
                "ENDANGERED": "is_endangered",
                "SPECIES_ID": "species_id",
                "Species3": "species_name",
                "Family3": "family_name",
                "Order3": "order_name",
                "Beak.Length_Culmen": "beak_length_culmen",
                "Beak.Length_Nares": "beak_length_nares",
                "Beak.Width": "beak_width",
                "Beak.Depth": "beak_depth",
                "Tarsus.Length": "tarsus_length",
                "Wing.Length": "wing_length",
                "Kipps.Distance": "kipps_distance",
                "Hand-Wing.Index": "hand_wing_index",
                "Tail.Length": "tail_length",
                "Mass": "mass",
            })
            .fillna({
                "is_endangered": 'N',
            })
    )
    return cleaned

transformed_df = transform_merge_dataset(merged_df)
transformed_df.head()

In [ ]:
AVONET_FEATURE_COLUMNS = [
    'beak_length_culmen',
    'beak_length_nares',
    'beak_width',
    'beak_depth',
    'tarsus_length',
    'wing_length',
    'kipps_distance',
    'hand_wing_index',
    'tail_length',
    'mass',
]

In [ ]:
# Create a density plot for each numeric feature
charts = []
for col in AVONET_FEATURE_COLUMNS:
    chart = (
        alt.Chart(transformed_df)
            .transform_density(col, as_=[col, 'density'])
            .mark_area(opacity=0.6)
            .encode(x=alt.X(col, title=col), y='density:Q')
            .properties(width=150, height=100)
    )
    charts.append(chart)
alt.concat(*charts, columns=3, title="Density Plots of Avonet Features")

### Feature Scaling

In [ ]:
def scale_feature_columns(data: pd.DataFrame, feature_columns: list[str]):
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data[feature_columns])

    data_scaled = data.copy()
    data_scaled[feature_columns] = scaled_features
    
    return data_scaled


scaled_df = scale_feature_columns(transformed_df, AVONET_FEATURE_COLUMNS)
scaled_df.head()

### UMAP Projection for Visualizations

In [ ]:
def compute_umap_projection(data: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    reducer = umap.UMAP(
        random_state=RANDOM_STATE,
        min_dist=0.1,
        n_neighbors=15,
        metric='euclidean'
    )
    embedding = reducer.fit_transform(data[feature_columns])

    data_umap = data.copy()
    data_umap['umap_component_1'] = embedding[:, 0]
    data_umap['umap_component_2'] = embedding[:, 1]
    return data_umap


In [ ]:
umap_df = compute_umap_projection(scaled_df, AVONET_FEATURE_COLUMNS)
umap_df.head()

### Visualizations

In [ ]:
def plot_umap(data: pd.DataFrame, color: alt.Color, title: str) -> alt.Chart:
    chart = (
    alt.Chart(data)
        .mark_circle(size=30)
        .encode(
            x=alt.X('umap_component_1'),
            y=alt.Y('umap_component_2'),
            color=color,
            tooltip=['family_name', 'order_name', 'is_endangered']
        )
        .properties(
            title=title,
            width=600,
            height=500
        )
    )
    return chart

In [ ]:
bird_order_plot = plot_umap(umap_df, alt.Color('order_name', legend=None), 'Bird Orders')
bird_family_plot = plot_umap(umap_df, alt.Color('family_name', legend=None), 'Bird Families')

plot = (
    (bird_order_plot | bird_family_plot)
        .interactive()
        .resolve_scale(x='shared', y='shared')
)
plot

In [ ]:
plot_umap(umap_df, alt.Color('is_endangered'), 'Endangered Status').interactive()

### K-Means Clustering
Evaluation Metrics:
1. Cluster cardinality is the number of examples per cluster. Plot the cluster cardinality for all clusters and investigate clusters that are major outliers.
2. Cluster magnitude is the sum of distances from all examples in a cluster to the cluster's centroid. Plot cluster magnitude for all clusters and investigate outliers.
3. Within-Cluster Sum of Squares (WCSS), also known as inertia in scikit-learn, measures how compact your clusters are.
   - Low WCSS -> data points are close to their centroids, clusters are tight and cohesive.
   - High WCSS -> clusters are spread out, possibly overlapping.

In [ ]:
def plot_elbow(data: pd.DataFrame, n_clusters: int, feature_columns: list[str]) -> alt.Chart:
    """Plots the elbow curve for KMeans clustering to determine the optimal number of clusters.
    """
    input_data = data[feature_columns]
    inertia = []
    for n in range(1, n_clusters + 1):
        kmeans = KMeans(n_clusters=n, random_state=RANDOM_STATE)
        kmeans.fit(input_data)
        inertia.append(kmeans.inertia_) 

    df = pd.DataFrame({
        'n_clusters': list(range(1, n_clusters + 1)),
        'inertia': inertia
    })
    chart = (
        alt.Chart(df)
            .mark_line(point=True)
            .encode(
                x=alt.X('n_clusters', title='Number of Clusters'),
                y=alt.Y('inertia', title='Inertia')
            )
            .properties(
                title='Elbow Method for Optimal Number of Clusters (K)',
                width=600,
                height=400
            )
    )
    return chart

plot_elbow(umap_df, 60, AVONET_FEATURE_COLUMNS)


In [ ]:
K = 15

In [ ]:
def compute_kmeans_clusters(data: pd.DataFrame, n_clusters: int, feature_columns: list[str]) -> pd.DataFrame:
    """Computes KMeans clustering and returns the data with cluster labels and cluster statistics.
    """
    input_data = data[feature_columns]
    kmeans = KMeans(
        n_clusters=n_clusters, 
        random_state=RANDOM_STATE
    )
    
    data_with_clusters = data.copy()
    data_with_clusters['cluster'] = kmeans.fit_predict(input_data)

    centroids = kmeans.cluster_centers_
    cardinality = pd.Series(kmeans.labels_).value_counts().sort_index()
    
    # Compute average distance and WCSS per cluster
    magnitudes = []
    wcss = []
    for i in range(len(centroids)):
        cluster_points = input_data[kmeans.labels_ == i]
        distances = np.linalg.norm(cluster_points - centroids[i], axis=1)

        magnitudes.append(distances.mean())
        wcss.append(np.sum(distances ** 2))
        
    cluster_stats = pd.DataFrame({
        'cluster': range(len(centroids)),
        'Cardinality': cardinality.values,
        'Magnitude (avg distance)': magnitudes,
        'WCSS (inertia)': wcss
    })

    return data_with_clusters, cluster_stats


clustered_df, cluster_stats = compute_kmeans_clusters(data=umap_df, n_clusters=K, feature_columns=AVONET_FEATURE_COLUMNS)
cluster_stats.head(100)

In [ ]:
def plot_cluster_stats(data: pd.DataFrame):
    """Plots cluster statistics: Cardinality, Magnitude, and WCSS."""
    color = alt.Color('cluster:N', legend=alt.Legend(title="Cluster"))

    base = (
        alt.Chart(data)
            .encode(
                x=alt.X('cluster:N', title='Cluster', axis=alt.Axis(labelAngle=0))  # keep labels horizontal
            )
    )

    cardinality_chart = (
        base.mark_bar()
        .encode(
            y=alt.Y('Cardinality:Q', title='Cardinality'),
            color=color
        )
        .properties(height=120)
    )

    magnitude_chart = (
        base.mark_bar()
        .encode(
            y=alt.Y('Magnitude (avg distance):Q', title='Magnitude (Avg Distance)'),
            color=color
        )
        .properties(height=120)
    )

    wcss_chart = (
        base.mark_bar()
        .encode(
            y=alt.Y('WCSS (inertia):Q', title='WCSS (Inertia)'),
            color=color
        )
        .properties(height=120)
    )

    final_chart = alt.vconcat(cardinality_chart, magnitude_chart, wcss_chart)
    return final_chart, color

In [ ]:
cluster_stats_plot, color = plot_cluster_stats(cluster_stats)
chart = (
    (plot_umap(clustered_df, color, 'Clustered') | cluster_stats_plot)
        .interactive()
        .resolve_scale(x='shared', y='shared', color='shared')
)

chart

In [ ]:
def categorize_clusters(df):
    summary = (
        df.groupby('cluster')['is_endangered']
        .agg(
            total='count',
            endangered_count=lambda x: (x == 'Y').sum(),
            non_endangered_count=lambda x: (x == 'N').sum()
        )
    )
    def classify(row):
        """Classify clusters based on endangered status composition."""
        if row['endangered_count'] == 0:
            return 'No Endangered'
        elif row['non_endangered_count'] == 0:
            return 'Only Endangered'
        else:
            return 'Mixed'

    summary['cluster_type'] = summary.apply(classify, axis=1)
    return summary.reset_index()

cluster_summary = categorize_clusters(clustered_df)
print(cluster_summary)

In [ ]:
clustered_df['cluster_type'] = clustered_df['cluster'].map(cluster_summary.set_index('cluster')['cluster_type'])
#clustered_df

In [ ]:
plot_umap(clustered_df, alt.Color('cluster_type', legend=alt.Legend(title="Cluster Type")), 'Cluster Types').interactive()

### Label Preparation

In [ ]:
# from sklearn.semi_supervised import LabelPropagation

# model = LabelPropagation(kernel="rbf", gamma=0.1, max_iter=5)

# y_semi = labels.copy()
# y_semi[y_semi == 'N'] = -1
# y_semi[y_semi == 'Y'] = 1
# y_semi = y_semi.astype(int)

# model.fit(scaled_data, y_semi)
# y_pred = model.transduction_